# Water Loss Analytics
## 02 — Data Cleaning
**Raden Satrio Hibatull Rasendriyo**

Pada tahap ini saya menyiapkan data audit untuk analisis SQL dan EDA. Notebook sebelumnya memuat 4.936 laporan, 2 laporan tanpa ID organisasi, 27 periode di luar 365–366 hari, dan 67 baris dengan setidaknya satu nilai kehilangan air atau NRW negatif. Angka tersebut merupakan hasil pemeriksaan awal, bukan jumlah baris yang otomatis harus dihapus.

Saya menyimpan seluruh laporan dalam tabel bertanda kualitas. Tabel analisis utama dibuat sebagai subset dengan aturan yang dijelaskan di bawah. Data sumber tetap tersedia untuk penelusuran.


### Cara menjalankan
Simpan di `water_loss_analytics/notebooks/`, pilih kernel Python yang sama dengan notebook 01, lalu klik **Run All**. Notebook ini membutuhkan pandas dan membaca `data/raw/ca-retailer-waterlossdata.csv` secara mandiri. Tidak perlu menjalankan notebook 01 dalam kernel yang sama.

Hasil disimpan ke `data/processed/` dan `outputs/tables/`. Menjalankan ulang notebook memperbarui file hasil dengan nama yang sama.

### Aturan kerja
- ID organisasi dan sistem disimpan sebagai teks. ID sistem hanya dirapikan spasi luar dan huruf besarnya; saya tidak menambahkan awalan CA atau menggabungkan ID yang berbeda secara otomatis.
- Volume menggunakan kolom `_AF`, sehingga satuannya acre-feet.
- Nilai negatif dengan besar absolut maksimal `1e-9` acre-feet diperlakukan sebagai nol numerik pada salinan analisis. Ini toleransi komputasi proyek, bukan ambang resmi. Semua penyesuaian dicatat.
- Nilai negatif di luar toleransi, angka tak terbaca, nilai tidak hingga, dan ketidaksesuaian dasar ditandai untuk ditinjau.
- Cakupan awal analisis utama: periode tahun kalender 2016–2024, identitas lengkap, tidak duplikat, angka utama memenuhi pemeriksaan dasar, serta status `Requirements Addressed` atau `Preliminarily Approved`. Status kedua tetap dibedakan untuk analisis sensitivitas.
- Periode lain dan 2025 tetap disimpan dalam tabel lengkap. Tahun 2025 hanya memiliki 31 laporan yang mulai pada tahun tersebut dalam snapshot ini; saya belum menganggap cakupannya sebanding dengan tahun sebelumnya.

Cakupan ini merupakan pilihan analisis proyek, bukan pernyataan bahwa laporan di luar cakupan salah. Skor validitas tidak diberi ambang minimum buatan; nantinya dipakai untuk menilai sensitivitas hasil.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = []
for base in [cwd, *cwd.parents]:
    candidates.extend([base, base / 'water_loss_analytics',
                       base / '02_Projects/water_loss_analytics'])
ROOT = next((p for p in candidates if
             (p / 'data/raw/ca-retailer-waterlossdata.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Simpan CSV di water_loss_analytics/data/raw dan notebook di notebooks.')
raw_path = ROOT / 'data/raw/ca-retailer-waterlossdata.csv'
source_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
raw = pd.read_csv(raw_path, dtype='string', encoding='utf-8-sig')
print(f'Data sumber: {len(raw):,} baris × {raw.shape[1]} kolom')

# Parameter cakupan proyek, dapat ditinjau kembali saat EDA.
ZERO_TOL_AF = 1e-9
START_YEAR, END_YEAR = 2016, 2024
MAIN_STATUSES = ['Requirements Addressed', 'Preliminarily Approved']


Data sumber: 4,936 baris × 190 kolom


### 1. Memilih kolom dan merapikan tipe data
Saya mengambil kolom yang diperlukan untuk identitas, periode, volume, ukuran jaringan, status, dan skor audit. Informasi kontak tidak dimasukkan ke tabel analisis. `source_row` menyimpan nomor baris data sumber (mulai 1, tidak termasuk header) agar setiap laporan mudah ditelusuri.


In [2]:
mapping = {
    'DWR_ORGANIZATION_ID': 'organization_id',
    'WATER_SUPPLIER_NAME': 'supplier_name',
    'PWS_ID_OR_OTHER_ID': 'system_id',
    'REPORTING_START_DATE': 'period_start',
    'REPORTING_END_DATE': 'period_end',
    'VOLUME_REPORTING_UNITS': 'original_volume_unit',
    'WS_WATER_SUPPLIED_VOL_AF': 'water_supplied_af',
    'WL_REAL_LOSSES_VOL_AF': 'real_losses_af',
    'WL_APPARENT_LOSSES_VOL_AF': 'apparent_losses_af',
    'NRW_NON_REVENUE_WATER_VOL_AF': 'non_revenue_water_af',
    'SD_NUM_SERVICE_CONN': 'service_connections',
    'SD_LENGTH_MAINS_MILES': 'mains_miles',
    'WATER_AUDIT_VALIDITY_SCORE': 'validity_score',
    'LATEST_STATUS': 'audit_status',
    'UPLOAD_VERSION': 'form_version',
}
missing = sorted(set(mapping) - set(raw.columns))
if missing:
    raise ValueError(f'Kolom sumber tidak tersedia: {missing}')
df = raw[list(mapping)].rename(columns=mapping).copy()
df = df.apply(lambda s: s.str.strip()).replace('', pd.NA)
df.insert(0, 'source_row', range(1, len(df) + 1))
df['system_id_original'] = df['system_id']
df['system_id'] = df['system_id'].str.upper()
# Format yang berbeda hanya ditandai, karena sumber mengizinkan ID selain PWS.
df['flag_nonstandard_system_id'] = ~df['system_id'].str.fullmatch(r'CA\d{7}', na=False)

volume_cols = ['water_supplied_af', 'real_losses_af',
               'apparent_losses_af', 'non_revenue_water_af']
num_cols = volume_cols + ['service_connections', 'mains_miles', 'validity_score']
conversion_log = []
for col in num_cols:
    original = df[col].copy()
    values = pd.to_numeric(original.str.replace(',', '', regex=False), errors='coerce')
    finite = pd.Series(np.isfinite(values.to_numpy(dtype=float, na_value=np.nan)), index=df.index)
    df[col] = values.where(finite)
    conversion_log.append({'column': col,
        'missing_source': int(original.isna().sum()),
        'invalid_nonempty': int((original.notna() & df[col].isna()).sum())})
for col in ['period_start', 'period_end']:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')
display(pd.DataFrame(conversion_log))
print('Organisasi unik terisi:', df['organization_id'].nunique())
display(df[['source_row', 'organization_id', 'system_id', 'period_start', 'water_supplied_af']].head())


,column,missing_source,invalid_nonempty
0,water_supplied_af,0,0
1,real_losses_af,0,0
2,apparent_losses_af,0,0
3,non_revenue_water_af,0,0
4,service_connections,0,0
5,mains_miles,0,0
6,validity_score,0,0


Organisasi unik terisi: 440


,source_row,organization_id,system_id,period_start,water_supplied_af
0,1,23,CA0110001,2016-01-01,37846.798555
1,2,23,CA0110001,2017-01-01,41086.555391
2,3,23,CA0110001,2018-01-01,42093.065741
3,4,23,CA0110001,2019-01-01,41451.347769
4,5,23,CA0110001,2020-01-01,44396.596889


### 2. Mencatat penyesuaian mendekati nol
Nilai seperti `-1.7e-16` sangat dekat dengan nol. Saya menyesuaikan hanya nilai negatif dalam toleransi yang telah ditetapkan, lalu menyimpan nilai sebelum dan sesudahnya. Nilai negatif yang lebih besar tetap dipertahankan dan masuk pemeriksaan kualitas.


In [3]:
adjustments = []
for col in volume_cols:
    tiny_negative = df[col].lt(0) & df[col].ge(-ZERO_TOL_AF)
    for idx in df.index[tiny_negative.fillna(False)]:
        adjustments.append({'source_row': int(df.at[idx, 'source_row']),
                            'column': col, 'original_value': float(df.at[idx, col]),
                            'adjusted_value': 0.0, 'reason': 'negative_within_numerical_tolerance'})
    df.loc[tiny_negative.fillna(False), col] = 0.0
adjustment_log = pd.DataFrame(adjustments, columns=[
    'source_row', 'column', 'original_value', 'adjusted_value', 'reason'])
df['flag_numeric_adjustment'] = df['source_row'].isin(adjustment_log['source_row'])
print('Jumlah sel yang disesuaikan:', len(adjustment_log))
display(adjustment_log.head(10))


Jumlah sel yang disesuaikan: 3


,source_row,column,original_value,adjusted_value,reason
0,77,real_losses_af,-1.734723e-16,0.0,negative_within_numerical_tolerance
1,78,real_losses_af,-2.949030e-17,0.0,negative_within_numerical_tolerance
2,79,real_losses_af,-1.587272e-16,0.0,negative_within_numerical_tolerance


### 3. Menandai kualitas dan cakupan
Pemeriksaan ini hanya memeriksa konsistensi dasar. Laporan yang lolos belum tentu bebas dari kesalahan pengukuran.

Saya juga menandai komponen kehilangan air yang melebihi pasokan dan NRW yang lebih kecil daripada jumlah kehilangan fisik serta semu. Selisih maksimal `1e-9` acre-feet ditoleransi untuk pembandingan numerik. Pemeriksaan ini bukan validasi neraca air lengkap.

Jumlah sambungan pecahan ditandai untuk peninjauan, tetapi tidak dibulatkan atau otomatis dikeluarkan karena definisi sumber belum menjelaskan perlakuan nilai pecahan.

Duplikasi diperiksa setelah normalisasi ID. Seluruh anggota kelompok duplikat ditahan untuk ditinjau, bukan dipilih salah satunya secara acak.


In [4]:
df['period_days'] = (df['period_end'] - df['period_start']).dt.days + 1
df['analysis_year'] = df['period_start'].dt.year.astype('Int64')
s, e = df['period_start'], df['period_end']
df['is_calendar_year'] = (s.dt.month.eq(1) & s.dt.day.eq(1) &
    e.dt.month.eq(12) & e.dt.day.eq(31) & s.dt.year.eq(e.dt.year))
key = ['organization_id', 'system_id', 'period_start', 'period_end']
df['flag_missing_identity'] = df[['organization_id','system_id']].isna().any(axis=1)
df['flag_invalid_period'] = s.isna() | e.isna() | e.lt(s)
df['flag_nonannual_duration'] = ~df['period_days'].between(365, 366)
complete_key = df[key].notna().all(axis=1)
df['flag_duplicate_key'] = complete_key & df.duplicated(key, keep=False)
df['flag_missing_numeric'] = df[num_cols].isna().any(axis=1)
df['flag_negative_volume'] = df[volume_cols].lt(0).any(axis=1)
df['flag_invalid_denominator'] = df[['water_supplied_af','service_connections','mains_miles']].le(0).any(axis=1)
df['flag_invalid_score'] = df['validity_score'].notna() & ~df['validity_score'].between(0,100)
df['flag_noninteger_connections'] = df['service_connections'].notna() & df['service_connections'].mod(1).ne(0)
loss_cols = ['real_losses_af','apparent_losses_af','non_revenue_water_af']
df['flag_volume_above_supply'] = df[loss_cols].gt(df['water_supplied_af'] + ZERO_TOL_AF, axis=0).any(axis=1)
df['flag_nrw_below_losses'] = (df['non_revenue_water_af'] + ZERO_TOL_AF).lt(
    df['real_losses_af'] + df['apparent_losses_af']).fillna(False)

quality_flags = ['flag_missing_identity','flag_invalid_period','flag_duplicate_key',
    'flag_missing_numeric','flag_negative_volume','flag_invalid_denominator',
    'flag_invalid_score',
    'flag_volume_above_supply','flag_nrw_below_losses']
# Semua flag memakai boolean biasa, tanpa nilai kosong.
flag_cols = [c for c in df if c.startswith('flag_')]
df[flag_cols] = df[flag_cols].fillna(False).astype(bool)
df['passes_basic_checks'] = ~df[quality_flags].any(axis=1)
df['in_year_scope'] = df['analysis_year'].between(START_YEAR, END_YEAR).fillna(False)
df['in_status_scope'] = df['audit_status'].isin(MAIN_STATUSES)
df['eligible_main'] = (df['passes_basic_checks'] & df['is_calendar_year'] &
                       df['in_year_scope'] & df['in_status_scope'])

reason_flags = df[quality_flags].copy()
reason_flags['outside_calendar_year_scope'] = ~df['is_calendar_year']
reason_flags['outside_year_range'] = ~df['in_year_scope']
reason_flags['outside_status_scope'] = ~df['in_status_scope']
df['exclusion_reasons'] = reason_flags.apply(
    lambda row: '; '.join(row.index[row.to_numpy(dtype=bool)]), axis=1)
quality_summary = df[flag_cols].sum().rename('row_count').rename_axis('flag').reset_index()
display(quality_summary)


,flag,row_count
0,flag_nonstandard_system_id,16
1,flag_numeric_adjustment,3
2,flag_missing_identity,2
3,flag_invalid_period,0
4,flag_nonannual_duration,27
5,flag_duplicate_key,0
6,flag_missing_numeric,0
7,flag_negative_volume,64
8,flag_invalid_denominator,0
9,flag_invalid_score,0


### 4. Membentuk tabel analisis dan memeriksa dampak filter
Tabel utama mencakup laporan tahun kalender yang lolos aturan proyek. Laporan berstatus `Preliminarily Approved` masih disertakan dan diberi penanda agar hasilnya dapat dibandingkan dengan subset `Requirements Addressed` saja.

Sistem yang tersedia belum tentu sama setiap tahun. Karena itu, penjumlahan volume tahunan belum boleh langsung ditafsirkan sebagai perbaikan atau penurunan efisiensi seluruh California. Pada EDA saya perlu memeriksa panel sistem yang konsisten.

ID gabungan hanya digunakan untuk pelacakan dalam dataset ini; perubahan organisasi atau ID sistem antarperiode tetap memerlukan peninjauan.


In [5]:
df['system_key'] = df['organization_id'].str.cat(df['system_id'], sep='|')
main = df.loc[df['eligible_main']].copy()
main['is_requirements_addressed'] = main['audit_status'].eq('Requirements Addressed')
# Rasio pada tingkat laporan. Agregasi nanti harus menghitung ulang dari jumlah volume.
main['nrw_share_pct'] = 100 * main['non_revenue_water_af'] / main['water_supplied_af']
main['real_loss_share_pct'] = 100 * main['real_losses_af'] / main['water_supplied_af']
main['real_loss_af_per_connection_year'] = main['real_losses_af'] / main['service_connections']

stages = [('Seluruh laporan', pd.Series(True, index=df.index)),
          ('Pemeriksaan dasar', df['passes_basic_checks']),
          ('Tahun kalender', df['is_calendar_year']),
          ('Rentang tahun proyek', df['in_year_scope']),
          ('Status pilihan proyek', df['in_status_scope'])]
keep = pd.Series(True, index=df.index)
rows = []
for label, condition in stages:
    previous = int(keep.sum())
    keep &= condition
    rows.append({'stage': label, 'remaining_rows': int(keep.sum()),
                 'excluded_at_stage': previous - int(keep.sum())})
filter_summary = pd.DataFrame(rows)
coverage = main.groupby('analysis_year').agg(
    audit_count=('source_row','size'), system_count=('system_key','nunique'),
    addressed_count=('is_requirements_addressed','sum'))
display(filter_summary)
display(coverage)
print('Status dalam tabel utama:')
display(main['audit_status'].value_counts())
print('ID dengan format selain CA + 7 digit di tabel utama:', int(main['flag_nonstandard_system_id'].sum()))


,stage,remaining_rows,excluded_at_stage
0,Seluruh laporan,4936,0
1,Pemeriksaan dasar,4870,66
2,Tahun kalender,3788,1082
3,Rentang tahun proyek,3757,31
4,Status pilihan proyek,3507,250


,audit_count,system_count,addressed_count
analysis_year,,,
2016,367,367,324
2017,393,393,345
2018,403,403,339
2019,401,401,338
2020,414,414,327
2021,413,413,271
2022,390,390,0
2023,395,395,0
2024,331,331,0


Status dalam tabel utama:


audit_status
Requirements Addressed    1944
Preliminarily Approved    1563
Name: count, dtype: Int64

ID dengan format selain CA + 7 digit di tabel utama: 4


### 5. Pemeriksaan akhir dan ekspor
`water_loss_all_flagged.csv` menyimpan semua laporan dengan kolom analisis dan penanda kualitas. `water_loss_analysis.csv` adalah subset utama untuk tahap SQL dan EDA. `water_loss_review.csv` berisi laporan di luar cakupan atau memiliki penanda yang perlu diperiksa; keberadaannya di file review tidak selalu berarti laporannya salah.

Tabel penyesuaian dan ringkasan filter disimpan sebagai jejak pengerjaan. File JSON mencatat parameter dan hash sumber agar hasil dapat ditelusuri ke snapshot CSV yang digunakan.


In [6]:
assert len(df) == len(raw), 'Ada baris hilang dari tabel lengkap.'
assert main['source_row'].is_unique
assert not main.duplicated(key).any()
assert main[num_cols].notna().all().all()
assert main[volume_cols].ge(0).all().all()
assert main['exclusion_reasons'].eq('').all()
assert int(keep.sum()) == len(main)
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == source_hash

processed = ROOT / 'data/processed'
tables = ROOT / 'outputs/tables'
processed.mkdir(parents=True, exist_ok=True)
tables.mkdir(parents=True, exist_ok=True)
review_mask = (~df['eligible_main'] | df['flag_nonstandard_system_id'] |
               df['flag_numeric_adjustment'] | df['flag_noninteger_connections'])
exports = {
    processed / 'water_loss_all_flagged.csv': df,
    processed / 'water_loss_analysis.csv': main,
    processed / 'water_loss_review.csv': df.loc[review_mask],
    tables / '02_numeric_adjustments.csv': adjustment_log,
    tables / '02_conversion_checks.csv': pd.DataFrame(conversion_log),
    tables / '02_quality_flags.csv': quality_summary,
    tables / '02_filter_summary.csv': filter_summary,
    tables / '02_year_coverage.csv': coverage.reset_index(),
}
for path, table in exports.items():
    table.to_csv(path, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    print(f'Tersimpan: {path.relative_to(ROOT)}')
metadata = dict(source_file=raw_path.name, source_sha256=source_hash,
    pandas_version=pd.__version__, zero_tolerance_af=ZERO_TOL_AF,
    start_year=START_YEAR, end_year=END_YEAR, calendar_year_only=True,
    main_statuses=MAIN_STATUSES, source_rows=len(raw), main_rows=len(main),
    numeric_adjustments=len(adjustment_log), quality_exclusion_flags=quality_flags,
    note='Project analysis scope; not official certification of data accuracy.')
(tables / '02_cleaning_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Selesai: {len(df):,} laporan tersimpan lengkap; {len(main):,} masuk tabel analisis utama.')


Tersimpan: data\processed\water_loss_all_flagged.csv
Tersimpan: data\processed\water_loss_analysis.csv
Tersimpan: data\processed\water_loss_review.csv
Tersimpan: outputs\tables\02_numeric_adjustments.csv
Tersimpan: outputs\tables\02_conversion_checks.csv
Tersimpan: outputs\tables\02_quality_flags.csv
Tersimpan: outputs\tables\02_filter_summary.csv
Tersimpan: outputs\tables\02_year_coverage.csv
Selesai: 4,936 laporan tersimpan lengkap; 3,507 masuk tabel analisis utama.


### Catatan untuk analisis berikutnya
- Hitung persentase NRW agregat dari total NRW dibagi total pasokan, bukan rata-rata sederhana persentase setiap laporan.
- Bandingkan hasil semua status pilihan dengan `Requirements Addressed` saja.
- Tinjau pengaruh skor validitas, penyesuaian mendekati nol, dan format ID yang berbeda.
- Periksa konsistensi identitas serta cakupan sistem sebelum menyimpulkan tren tahunan.
- Metrik per sambungan di notebook ini merupakan rasio tahunan deskriptif; bukan indikator resmi gallons/connection/day atau standar penilaian kebocoran.

Tahap berikutnya adalah analisis SQL dan EDA berdasarkan tabel hasil cleaning, dengan mencatat keterbatasan cakupannya dalam dashboard dan PPT.
